# Predator-Prey Disease Dynamics (Eco-Epidemiology)
## Capstone Project Notebook

Eco-epidemiology is a field that combines **population ecology** (predator-prey dynamics) with **epidemiology** (disease transmission). The central question is: *how does an infectious disease in one or more species alter the classical predator-prey relationship?*

In a standard Lotka-Volterra system, predators and prey coexist in characteristic oscillations. But what happens when a disease enters this system?

- **Disease in the prey** may make infected individuals easier to catch, providing predators with an "easy meal" but also exposing them to potential parasites. The disease can act as an additional mortality factor that competes with predation.
- **Disease in the predator** may reduce hunting efficiency of sick individuals, allowing prey populations to grow. If the disease is density-dependent, it may regulate predator populations in ways that stabilise or destabilise the system.

These interactions create rich dynamics that neither pure ecology nor pure epidemiology can capture alone. Eco-epidemiological models reveal phenomena such as:

- **Disease-mediated coexistence** — disease can prevent competitive exclusion
- **Hydra effect** — increased mortality (from disease) paradoxically increases population size
- **Disease-induced extinction** — disease can tip a predator-prey system into collapse
- **Selective predation** — predators preferentially consuming infected prey can reduce disease prevalence, acting as a "biological control"

### Mathematical Framework

These models typically extend the Lotka-Volterra system by splitting one species into epidemiological compartments (Susceptible, Infected, Recovered). The resulting ODE systems have 3–5 state variables and exhibit complex bifurcation behaviour including limit cycles, Hopf bifurcations, and chaos.

### In this notebook you will:
1. Review the **Lotka-Volterra** predator-prey model
2. Extend it with **disease in the prey** population (SI-Predator model)
3. Implement **disease in the predator** population (Prey-SI model)
4. Compare both models and analyse how disease alters population cycles
5. Explore parameter sensitivity
6. Lay the groundwork for required project extensions

---

## 0 · Setup

We import NumPy for numerical integration and Matplotlib for plotting. We define three colours for the key populations: green for susceptible/prey, red for infected, and purple for predators. These will be used consistently across all plots.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 12})
C_S = '#2ecc71'; C_I = '#e74c3c'; C_P = '#8e44ad'

---
## 1 · Classical Lotka-Volterra

Before introducing disease, we establish the baseline: the classical Lotka-Volterra predator-prey model. This deterministic ODE system produces perfectly periodic oscillations — prey grow exponentially, predators follow with a lag, overexploit the prey, then decline, allowing prey to recover. The amplitude and period of these cycles depend on the parameters and initial conditions.

$$\frac{dx}{dt} = \alpha x - \beta x y, \qquad \frac{dy}{dt} = \delta x y - \gamma y$$

where $x$ = prey, $y$ = predators. This baseline will let us see exactly how disease alters the predator-prey dynamics in the models that follow.

In [ ]:
def simulate_lv(x0, y0, alpha, beta, delta, gamma, T, dt=0.01):
    steps = int(T / dt) + 1
    t = np.linspace(0, T, steps)
    x, y = np.zeros(steps), np.zeros(steps)
    x[0], y[0] = x0, y0
    for k in range(1, steps):
        dx = alpha * x[k-1] - beta * x[k-1] * y[k-1]
        dy = delta * x[k-1] * y[k-1] - gamma * y[k-1]
        x[k] = max(0, x[k-1] + dt * dx)
        y[k] = max(0, y[k-1] + dt * dy)
    return t, x, y

t_lv, x_lv, y_lv = simulate_lv(40, 9, 1.0, 0.1, 0.075, 1.5, 30)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_lv, x_lv, color=C_S, lw=2, label='Prey')
ax.plot(t_lv, y_lv, color=C_P, lw=2, label='Predators')
ax.set_xlabel('Time'); ax.set_ylabel('Population')
ax.set_title('Classical Lotka-Volterra', fontweight='bold'); ax.legend()
plt.tight_layout(); plt.show()

---
## 2 · Model 1: Disease in the Prey (SI-Predator)

We split the prey population into **Susceptible** ($S$) and **Infected** ($I$) compartments. Predators ($P$) hunt both types but may prefer infected prey — a common ecological observation, since sick animals are often slower, weaker, or less vigilant.

The key new parameters are:
- $\lambda$ — disease transmission rate among prey (frequency-dependent)
- $\mu$ — additional mortality from disease (on top of natural death and predation)
- $\beta_I > \beta_S$ — infected prey are easier to catch (selective predation)
- $\delta_I < \delta_S$ — but diseased prey may provide less nutritional value

This selective predation creates an interesting feedback: predators preferentially remove infected individuals, which *reduces* disease prevalence in the prey population. The predator acts as a **biological control agent** for the disease. However, if the predator becomes too efficient at removing infected prey, the disease cannot persist — and the system reverts to classical Lotka-Volterra dynamics.

In [ ]:
def simulate_si_predator(S0, I0, P0, alpha, K, lam, mu,
                           beta_S, beta_I, delta_S, delta_I, gamma,
                           T, dt=0.01):
    steps = int(T / dt) + 1
    t = np.linspace(0, T, steps)
    S, I, P = [np.zeros(steps) for _ in range(3)]
    S[0], I[0], P[0] = S0, I0, P0
    for k in range(1, steps):
        s, i, p = S[k-1], I[k-1], P[k-1]
        dS = alpha * s * (1 - (s + i) / K) - lam * s * i - beta_S * s * p
        dI = lam * s * i - mu * i - beta_I * i * p
        dP = delta_S * beta_S * s * p + delta_I * beta_I * i * p - gamma * p
        S[k] = max(0, s + dt * dS)
        I[k] = max(0, i + dt * dI)
        P[k] = max(0, p + dt * dP)
    return t, S, I, P

# Baseline parameters
t1, S1, I1, P1 = simulate_si_predator(
    S0=40, I0=5, P0=9,
    alpha=1.0, K=100, lam=0.01, mu=0.3,
    beta_S=0.05, beta_I=0.1, delta_S=0.08, delta_I=0.04,
    gamma=1.0, T=60)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t1, S1, color=C_S, lw=2, label='Susceptible prey')
ax.plot(t1, I1, color=C_I, lw=2, label='Infected prey')
ax.plot(t1, P1, color=C_P, lw=2, label='Predators')
ax.set_xlabel('Time'); ax.set_ylabel('Population')
ax.set_title('Model 1: SI-Predator (disease in prey)', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()

---
## 3 · Model 2: Disease in the Predator (Prey-SI)

Now the disease afflicts the **predator** population instead. Predators are split into Healthy ($H$) and Diseased ($D$) compartments. Diseased predators hunt less efficiently ($\beta_D < \beta_H$) and suffer additional mortality ($\mu$).

The key ecological insight: disease in the predator population **releases prey from predation pressure**. If enough predators become sick and die, prey populations can boom — which may then support a recovery of healthy predators, creating complex oscillatory dynamics.

$$\frac{dX}{dt} = \alpha X \left(1 - \frac{X}{K}\right) - \beta_H X H - \beta_D X D$$

$$\frac{dH}{dt} = \delta \beta_H X H - \lambda H D - \gamma H$$

$$\frac{dD}{dt} = \delta_D \beta_D X D + \lambda H D - (\gamma + \mu) D$$

where $X$ = prey, $H$ = healthy predators, $D$ = diseased predators. Disease transmission among predators is density-dependent ($\lambda H D$), so larger predator populations experience more transmission.

In [ ]:
def simulate_prey_si(X0, H0, D0, alpha, K, beta_H, beta_D,
                       delta, delta_D, lam, gamma, mu, T, dt=0.01):
    steps = int(T / dt) + 1
    t = np.linspace(0, T, steps)
    X, H, D = [np.zeros(steps) for _ in range(3)]
    X[0], H[0], D[0] = X0, H0, D0
    for k in range(1, steps):
        x, h, d = X[k-1], H[k-1], D[k-1]
        dX = alpha * x * (1 - x / K) - beta_H * x * h - beta_D * x * d
        dH = delta * beta_H * x * h - lam * h * d - gamma * h
        dD = delta_D * beta_D * x * d + lam * h * d - (gamma + mu) * d
        X[k] = max(0, x + dt * dX)
        H[k] = max(0, h + dt * dH)
        D[k] = max(0, d + dt * dD)
    return t, X, H, D

t2, X2, H2, D2 = simulate_prey_si(
    X0=50, H0=10, D0=2,
    alpha=1.0, K=100, beta_H=0.08, beta_D=0.03,
    delta=0.08, delta_D=0.04, lam=0.02, gamma=0.8, mu=0.3, T=60)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t2, X2, color=C_S, lw=2, label='Prey')
ax.plot(t2, H2, color=C_P, lw=2, label='Healthy predators')
ax.plot(t2, D2, color=C_I, lw=2, label='Diseased predators')
ax.set_xlabel('Time'); ax.set_ylabel('Population')
ax.set_title('Model 2: Prey-SI (disease in predators)', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()

---
## 4 · Comparing Both Models

To understand how disease location matters, we compare all three systems side by side: the classical Lotka-Volterra (no disease), Model 1 (disease in prey), and Model 2 (disease in predators). For Models 1 and 2, we plot the **total prey** and **total predator** populations (summing across disease compartments) to make a fair comparison, with the infected subpopulation shown as a dashed line.

Key differences to look for:
- Disease tends to **damp oscillations** compared to the undamped Lotka-Volterra cycles
- Disease in prey reduces total prey but may stabilise the system
- Disease in predators releases prey from predation pressure, potentially increasing prey abundance

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4))

# LV baseline
ax1.plot(t_lv, x_lv, color=C_S, lw=2); ax1.plot(t_lv, y_lv, color=C_P, lw=2)
ax1.set_title('Classic LV\n(no disease)', fontweight='bold')

# Model 1
ax2.plot(t1, S1+I1, color=C_S, lw=2); ax2.plot(t1, P1, color=C_P, lw=2)
ax2.plot(t1, I1, color=C_I, lw=1.5, ls='--')
ax2.set_title('Model 1: Disease in prey', fontweight='bold')

# Model 2
ax3.plot(t2, X2, color=C_S, lw=2); ax3.plot(t2, H2+D2, color=C_P, lw=2)
ax3.plot(t2, D2, color=C_I, lw=1.5, ls='--')
ax3.set_title('Model 2: Disease in predators', fontweight='bold')

for ax in [ax1, ax2, ax3]:
    ax.set_xlabel('Time'); ax.set_ylabel('Population')
plt.tight_layout(); plt.show()

---
## 5 · Parameter Sensitivity

Three parameters have especially strong effects on the eco-epidemiological dynamics:

- **Disease transmission rate** $\lambda$ (left) — controls how fast the disease spreads among prey. Low $\lambda$ means disease is rare and the system behaves like classical Lotka-Volterra. High $\lambda$ means most prey become infected, increasing predator food supply (since infected prey are easier to catch) but also increasing disease-induced mortality.
- **Disease mortality** $\mu$ (centre) — controls how lethal the disease is. High virulence ($\mu$) kills infected prey quickly, limiting disease spread but also reducing the prey population. Low virulence allows infected prey to persist longer, creating a larger pool of easy targets for predators.
- **Selective predation** $\beta_I$ (right) — controls how strongly predators prefer infected prey. High selective predation means predators rapidly remove infected individuals, suppressing the disease. This "healthy herding" effect can prevent epidemics entirely if predation on infected prey is strong enough.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Vary disease transmission
for lam_v, c in zip([0.005, 0.01, 0.03], ['#3498db', '#e67e22', '#e74c3c']):
    _, S_v, I_v, P_v = simulate_si_predator(
        40, 5, 9, 1.0, 100, lam_v, 0.3, 0.05, 0.1, 0.08, 0.04, 1.0, 60)
    axes[0].plot(I_v, color=c, lw=1.5, label=f'$\\lambda$={lam_v}')
axes[0].set_title('Vary Transmission Rate'); axes[0].set_ylabel('Infected prey')

# Vary disease mortality
for mu_v, c in zip([0.1, 0.3, 0.8], ['#3498db', '#e67e22', '#e74c3c']):
    _, S_v, I_v, P_v = simulate_si_predator(
        40, 5, 9, 1.0, 100, 0.01, mu_v, 0.05, 0.1, 0.08, 0.04, 1.0, 60)
    axes[1].plot(I_v, color=c, lw=1.5, label=f'$\\mu$={mu_v}')
axes[1].set_title('Vary Disease Mortality'); axes[1].set_ylabel('Infected prey')

# Vary selective predation (beta_I)
for bi, c in zip([0.05, 0.1, 0.3], ['#3498db', '#e67e22', '#e74c3c']):
    _, S_v, I_v, P_v = simulate_si_predator(
        40, 5, 9, 1.0, 100, 0.01, 0.3, 0.05, bi, 0.08, 0.04, 1.0, 60)
    axes[2].plot(I_v, color=c, lw=1.5, label=f'$\\beta_I$={bi}')
axes[2].set_title('Vary Selective Predation'); axes[2].set_ylabel('Infected prey')

for ax in axes:
    ax.set_xlabel('Time'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

---
## 6 · Your Tasks

1. **Model Implementation**: Find and implement at least two different predator-prey models with disease dynamics (two are provided above as starting points).
2. **Tracking & Visualisation**: Track population sizes over time and show how disease alters traditional cycles.

### Extensions to explore
- Phase plane analysis ($S$ vs $I$, prey vs predator)
- Stability analysis: find equilibria and determine their stability
- SIR variant: add recovery from disease
- Seasonal forcing of disease transmission
- Compare deterministic ODE with stochastic simulation

### Discussion points
- How does disease change classic predator-prey oscillations?
- Can disease stabilise or destabilise coexistence?
- What emergent patterns appear (disease-induced extinction, etc.)?
- Present alternative models you found but did not implement

In [ ]:
# TODO: Delete this cell

---
## Recommended Reading & Journal Club

**1. Anderson, R. M. & May, R. M. (1986)** *The invasion, persistence and spread of infectious diseases within animal and plant communities.* Phil. Trans. R. Soc. B, 314, 533–570.
→ Foundational paper combining epidemiology with ecology.

**2. Chattopadhyay, J. & Arino, O. (1999)** *A predator-prey model with disease in the prey.* Nonlinear Analysis, 36(6), 747–766. [DOI](https://doi.org/10.1016/S0362-546X(98)00126-6)
→ Mathematical analysis of the SI-predator model.

**3. Hethcote, H. W. et al. (2004)** *A predator-prey model with infected prey.* Theoretical Population Biology, 66(3), 259–268.
→ Stability and bifurcation analysis.

**4. Venturino, E. (2002)** *Epidemics in predator-prey models: disease in the predators.* IMA J. Math. Applied in Medicine and Biology, 19(3), 185–205.
→ Covers Model 2 (disease in predators).